# Practice: Building an ML Quality Gate for a Breast Cancer Classifier

**Dataset:** the [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) dataset, loaded via `sklearn.datasets.load_breast_cancer`. This is a real dataset of 569 breast tumor biopsies, each described by 30 real-valued features computed from digitized images of cell nuclei (radius, texture, perimeter, concavity, etc.), with a binary label: `0 = malignant`, `1 = benign`.

**Scenario:** You're the QA engineer embedded in an ML team that's about to ship a tumor-triage classifier. The data scientists handed you a trained model and said "it's 96% accurate, ship it." Your job — drawing on your SQA background — is to build the automated **quality gate** that a real CI/CD pipeline would run before any model is allowed to deploy: a set of pass/fail checks on the *data*, the *model's behavior*, its *robustness to drift*, and its *reproducibility*. A single misleading metric (like overall accuracy) should never be the only thing standing between a model and production.

**What you should produce** (five functions, filled in where you see `# TODO`):

1. `validate_schema(df)` — a data-validation check that fails loudly if incoming data doesn't match the expected contract: exact column set, no nulls, and no physically impossible values (every one of these 30 features is a non-negative measurement, so any negative value is a bug upstream).
2. `evaluate_model(model, X_test, y_test, min_recall_malignant=0.95)` — compute accuracy, precision, recall, and F1, **and** explicitly check recall on the malignant class against a threshold. Missing a malignant tumor (false negative) is far costlier than a false alarm, so overall accuracy alone is not an acceptable release gate here.
3. `detect_drift(reference, new, feature, alpha=0.05)` — a two-sample Kolmogorov–Smirnov test (`scipy.stats.ks_2samp`) that flags whether a single feature's distribution in new ("production") data has shifted away from the training reference distribution.
4. `check_reproducibility(train_fn, X_train, y_train, X_test, n_runs=2)` — train the same model twice with the same random seed and confirm the predictions are bit-for-bit identical. A model that can't reproduce its own results can't be debugged or audited.
5. `run_quality_gate(...)` — call all of the above against a held-out test set *and* a simulated "production" batch (provided for you, with an intentionally injected feature shift), and print a clear PASS/FAIL report, one line per check, plus an overall verdict.

Work through the starter cells below, then compare against the solution at the bottom (try it yourself first!).

## Setup

Imports and data loading — nothing to fill in here.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy.stats import ks_2samp

RANDOM_STATE = 42

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

print(df.shape)
print("Target classes:", dict(enumerate(data.target_names)))  # 0=malignant, 1=benign
print(df["target"].value_counts())
df.head()

Split into train/test. This is the *reference* distribution our quality gate will check everything against.

In [ ]:
FEATURE_COLUMNS = [c for c in df.columns if c != "target"]

X = df[FEATURE_COLUMNS]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

X_train.shape, X_test.shape

## Part 1 — Data validation

Fill in `validate_schema`. It should check, against a reference schema built from the training data:

- the incoming DataFrame has exactly the expected columns (no missing, no extras)
- no nulls anywhere
- every feature value is `>= 0` (all 30 features here are physical measurements — negative values mean an upstream bug, not a valid tumor)

Return a list of human-readable violation strings (empty list = valid).

In [ ]:
def validate_schema(df_to_check, expected_columns=FEATURE_COLUMNS):
    \"\"\"Return a list of violation messages; an empty list means the data is valid.\"\"\"
    violations = []

    # TODO: check that df_to_check has exactly `expected_columns` (missing + extra columns)

    # TODO: check for any null values and report which columns contain them

    # TODO: check that every value in the feature columns is >= 0

    return violations


# quick sanity checks once you've implemented it above
print("Valid data violations:", validate_schema(X_test))

bad_batch = X_test.copy()
bad_batch.loc[bad_batch.index[0], "mean radius"] = -5.0
bad_batch.loc[bad_batch.index[1], "mean texture"] = np.nan
print("Corrupted data violations:", validate_schema(bad_batch))

## Part 2 — Evaluation beyond accuracy

Train the baseline model (`LogisticRegression`), then fill in `evaluate_model`. It must return a dict with `accuracy`, `precision`, `recall`, `f1`, `confusion_matrix`, AND a boolean `passes_recall_gate` that is `True` only if recall on the **malignant** class (label `0`) is `>= min_recall_malignant`.

Think about why you can't just use `recall_score`'s default — which class does it score by default, and is that the one that matters here?

In [ ]:
baseline_model = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)


def evaluate_model(model, X_eval, y_eval, min_recall_malignant=0.95):
    \"\"\"Return a dict of metrics, including a pass/fail recall gate on the malignant class.\"\"\"
    preds = model.predict(X_eval)

    metrics = {}
    # TODO: metrics["accuracy"] = ...
    # TODO: metrics["precision"] = ...  (use pos_label=0 for the malignant class)
    # TODO: metrics["recall"] = ...     (use pos_label=0 for the malignant class)
    # TODO: metrics["f1"] = ...         (use pos_label=0 for the malignant class)
    # TODO: metrics["confusion_matrix"] = ...
    # TODO: metrics["passes_recall_gate"] = metrics["recall"] >= min_recall_malignant

    return metrics


evaluate_model(baseline_model, X_test, y_test)

## Part 3 — Drift detection

In production, the real-world distribution of incoming data can silently shift away from what the model was trained on (new imaging equipment, a different patient population, a sensor recalibration...). Fill in `detect_drift` using a two-sample KS test to catch this on a single feature.

To test it, we build a **simulated production batch**: we take the real test set and *inject* an artificial shift into one feature (`mean radius`). This injection is synthetic on purpose — we need a case with *known* ground-truth drift to confirm the detector actually works, which a purely real, unmodified dataset can't give us on demand.

In [ ]:
def detect_drift(reference, new, feature, alpha=0.05):
    \"\"\"Run a KS test comparing `feature` in `reference` vs `new`.

    Return a dict: {"statistic": ..., "p_value": ..., "drifted": bool}
    `drifted` is True when the p-value is below `alpha` (distributions differ significantly).
    \"\"\"
    # TODO: stat, p_value = ks_2samp(reference[feature], new[feature])
    # TODO: return {"statistic": stat, "p_value": p_value, "drifted": p_value < alpha}
    pass


# simulated production batch: real test data, with an injected shift on one feature
production_batch = X_test.copy()
production_batch["mean radius"] = production_batch["mean radius"] * 1.15 + 2.0  # injected drift

print("Clean batch  (should NOT flag drift):", detect_drift(X_train, X_test, "mean radius"))
print("Shifted batch (SHOULD flag drift):   ", detect_drift(X_train, production_batch, "mean radius"))

## Part 4 — Reproducibility

A model that gives different answers on different runs of the *same* training code with the *same* seed is impossible to audit or debug reliably. Fill in `check_reproducibility`: train a fresh model `n_runs` times with `train_fn`, predict on `X_test`, and confirm every run's predictions are identical to the first.

In [ ]:
def train_fn(X_tr, y_tr):
    model = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
    model.fit(X_tr, y_tr)
    return model


def check_reproducibility(train_fn, X_tr, y_tr, X_eval, n_runs=2):
    \"\"\"Return True if every run's predictions on X_eval are identical.\"\"\"
    # TODO: train `n_runs` models via train_fn(X_tr, y_tr), predict on X_eval each time,
    #       and compare every run's predictions to the first run's with np.array_equal
    pass


check_reproducibility(train_fn, X_train, y_train, X_test)

## Part 5 — Assemble the quality gate

Fill in `run_quality_gate` to tie everything together. It should run, in order:

1. `validate_schema` on the production batch
2. `evaluate_model` on the clean test set (including the recall gate)
3. `detect_drift` on `"mean radius"` comparing train vs. the production batch
4. `check_reproducibility`

...then print one `PASS`/`FAIL` line per check, and a final overall verdict (PASS only if every check passes — for drift, "PASS" means *no* unexpected drift was found on a clean comparison, but here we're deliberately feeding it the shifted batch so this check *should* report FAIL, demonstrating the gate catching a real problem before deploy).

In [ ]:
def run_quality_gate(model, X_train, y_train, X_test, y_test, production_batch):
    print("=== ML Quality Gate Report ===")

    # TODO 1: schema_violations = validate_schema(production_batch)
    # TODO:   print a PASS/FAIL line based on whether schema_violations is empty

    # TODO 2: metrics = evaluate_model(model, X_test, y_test)
    # TODO:   print a PASS/FAIL line based on metrics["passes_recall_gate"]

    # TODO 3: drift_result = detect_drift(X_train, production_batch, "mean radius")
    # TODO:   print a PASS/FAIL line — note this is EXPECTED to report drift (FAIL) here

    # TODO 4: reproducible = check_reproducibility(train_fn, X_train, y_train, X_test)
    # TODO:   print a PASS/FAIL line

    # TODO: combine into an overall verdict and return it (bool)
    pass


run_quality_gate(baseline_model, X_train, y_train, X_test, y_test, production_batch)

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Approach:** each check is a small, independent function that returns structured data (not just a print), so the quality gate can compose them and decide pass/fail itself — this mirrors how a real CI assertion library is built. Note what actually happens when you run this on the real model: **accuracy is ~95.8%, which looks great, but recall on the malignant class is only ~90.6% — below the 0.95 gate.** That's the whole point of Part 2: a single aggregate metric hid a failure mode that matters most for this problem (missed cancers).

```python
def validate_schema(df_to_check, expected_columns=FEATURE_COLUMNS):
    violations = []

    actual_columns = set(df_to_check.columns)
    expected = set(expected_columns)
    missing = expected - actual_columns
    extra = actual_columns - expected
    if missing:
        violations.append(f"Missing columns: {sorted(missing)}")
    if extra:
        violations.append(f"Unexpected extra columns: {sorted(extra)}")

    present_expected = [c for c in expected_columns if c in df_to_check.columns]
    null_counts = df_to_check[present_expected].isnull().sum()
    for col, count in null_counts[null_counts > 0].items():
        violations.append(f"Column '{col}' has {count} null value(s)")

    for col in present_expected:
        negative_count = (df_to_check[col] < 0).sum()
        if negative_count > 0:
            violations.append(f"Column '{col}' has {negative_count} negative value(s)")

    return violations


def evaluate_model(model, X_eval, y_eval, min_recall_malignant=0.95):
    preds = model.predict(X_eval)

    metrics = {
        "accuracy": accuracy_score(y_eval, preds),
        "precision": precision_score(y_eval, preds, pos_label=0),
        "recall": recall_score(y_eval, preds, pos_label=0),
        "f1": f1_score(y_eval, preds, pos_label=0),
        "confusion_matrix": confusion_matrix(y_eval, preds),
    }
    metrics["passes_recall_gate"] = metrics["recall"] >= min_recall_malignant
    return metrics


def detect_drift(reference, new, feature, alpha=0.05):
    stat, p_value = ks_2samp(reference[feature], new[feature])
    return {"statistic": stat, "p_value": p_value, "drifted": p_value < alpha}


def check_reproducibility(train_fn, X_tr, y_tr, X_eval, n_runs=2):
    first_preds = None
    for _ in range(n_runs):
        model = train_fn(X_tr, y_tr)
        preds = model.predict(X_eval)
        if first_preds is None:
            first_preds = preds
        elif not np.array_equal(first_preds, preds):
            return False
    return True


def run_quality_gate(model, X_train, y_train, X_test, y_test, production_batch):
    print("=== ML Quality Gate Report ===")
    results = {}

    schema_violations = validate_schema(production_batch)
    results["schema"] = len(schema_violations) == 0
    print(f"[{'PASS' if results['schema'] else 'FAIL'}] Schema validation"
          + ("" if results["schema"] else f" — {schema_violations}"))

    metrics = evaluate_model(model, X_test, y_test)
    results["recall_gate"] = metrics["passes_recall_gate"]
    print(f"[{'PASS' if results['recall_gate'] else 'FAIL'}] Recall gate "
          f"(malignant recall={metrics['recall']:.3f}, accuracy={metrics['accuracy']:.3f})")

    drift_result = detect_drift(X_train, production_batch, "mean radius")
    results["no_unexpected_drift"] = not drift_result["drifted"]
    print(f"[{'PASS' if results['no_unexpected_drift'] else 'FAIL'}] Drift check on 'mean radius' "
          f"(p={drift_result['p_value']:.2e}) — expected to FAIL: this batch has injected drift")

    results["reproducible"] = check_reproducibility(train_fn, X_train, y_train, X_test)
    print(f"[{'PASS' if results['reproducible'] else 'FAIL'}] Reproducibility")

    overall = all(results.values())
    print(f"\nOVERALL: {'PASS' if overall else 'FAIL'}")
    return overall


run_quality_gate(baseline_model, X_train, y_train, X_test, y_test, production_batch)
```

**Why this matters for an SQA → ML QA transition:** the shape is identical to a traditional test suite (arrange/act/assert, fixtures, pass/fail reporting) — what's different is *what* you assert on. Schema validation is like input-contract testing. The recall gate is a reminder that "accuracy" is often the wrong acceptance criterion; you need to pick the metric that matches the real cost of errors. Drift detection is a test you run continuously in production, not just once at release — it's the ML equivalent of a canary/health check. And reproducibility is the ML version of "flaky test" hygiene: non-determinism in training is a bug, not a quirk.

</details>